# Specimen (PMDCo): from data entry to RDF

This notebook shows how to describe a physical specimen (its name, mass, and
chemical composition) and convert that description into a standardised,
machine-readable RDF graph following the
[Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/).

**You only need to edit one file:** `docs/example.input.json`.  Everything
else is automatic.

---

## What the notebook does

```
example.input.json
  │  your specimen name, mass, and element fractions
  │
  ▼
Transform
  │  converts your plain JSON into a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
SPARQL inspect    →  shows key values from the graph
```

---

## Environment setup

If you are viewing this notebook on GitHub and want to run it locally, clone
the repository first so that all schema files and example inputs are present:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
```

Then create a virtual environment and start Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

Open this notebook from the `docs/` subfolder inside JupyterLab.

In [1]:
%pip install -q jsonata-python rdflib pyyaml pyshacl

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE      = pathlib.Path().resolve()              # docs/
SPECIMEN  = HERE.parent                           # schemas/specimen/PMDCo/
SCHEMAS   = SPECIMEN.parents[1]                   # schemas/
CHEM_COMP = SCHEMAS / "chemical-composition" / "PMDCo"

specimen_schema = Schema(SPECIMEN)
comp_schema     = Schema(CHEM_COMP)

# Base IRI for this dataset — replace with your own namespace in production.
# Relative node IDs in the OO-LD document (e.g. "specimen-316l-tensile-bar-1")
# will be resolved against this base when building the RDF graph.
BASE = "https://example.org/"

---
## Step 1: Describe your specimen

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `specimen_name` | yes | A name or ID for the specimen |
| `elements` | yes | List of `{symbol, value, unit}` (same format as the chemical-composition schema) |
| `mass_value` | no | Mass of the specimen as a positive number |
| `mass_unit` | no | `"g"` (gram), `"kg"` (kilogram), or `"mg"` (milligram) |
| `specimen_id` | no | Custom IRI slug (auto-derived from `specimen_name` if omitted) |
| `comp_id` | no | Custom IRI slug for the composition node (auto-derived if omitted) |

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "specimen_name": "316L Tensile Bar #1",
  "mass_value": 50.3,
  "mass_unit": "g",
  "elements": [
    {
      "symbol": "Fe",
      "value": 65.345,
      "unit": "mass%"
    },
    {
      "symbol": "Cr",
      "value": 17.0,
      "unit": "mass%"
    },
    {
      "symbol": "Ni",
      "value": 12.0,
      "unit": "mass%"
    },
    {
      "symbol": "Mo",
      "value": 2.5,
      "unit": "mass%"
    },
    {
      "symbol": "Mn",
      "value": 1.8,
      "unit": "mass%"
    },
    {
      "symbol": "Si",
      "value": 0.5,
      "unit": "mass%"
    },
    {
      "symbol": "N",
      "value": 0.1,
      "unit": "mass%"
    },
    {
      "symbol": "C",
      "value": 0.03,
      "unit": "mass%"
    },
    {
      "symbol": "P",
      "value": 0.045,
      "unit": "mass%"
    },
    {
      "symbol": "S",
      "value": 0.03,
      "unit": "mass%"
    }
  ]
}


---
## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document, the intermediate
format that carries both the data and its ontology mapping.

The specimen schema is built on top of the
[chemical composition schema](../../chemical-composition/PMDCo/README.md):
the composition part of the output is produced by that schema's converter,
which is the authoritative source for element IRIs and node naming conventions.
The two results are then merged into a single document.

In [4]:
# Specimen envelope (name, mass)
specimen_doc = specimen_schema.transform(simplified)


In [5]:
# Chemical composition (delegated to the composition schema)
comp_input = {
    "material_name": simplified["specimen_name"],
    "material_id":   specimen_doc["id"],
    "elements":      simplified["elements"],
}
if "comp_id" in simplified:
    comp_input["comp_id"] = simplified["comp_id"]

comp_doc = comp_schema.transform(comp_input)

# The composition converter embeds the material as a labelled node (standalone use).
# Here the material is the specimen itself, so we replace it with the specimen IRI.
comp_doc["quality_of"] = specimen_doc["id"]


In [6]:
# ── Merge into the final OO-LD document ───────────────────────────────
oold_doc = {**specimen_doc, "has_composition": comp_doc}

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/specimen/PMDCo/#v1.0.0",
  "type": "bfo:0000030",
  "id": "specimen-316l-tensile-bar-1",
  "label": "316L Tensile Bar #1",
  "mass": {
    "type": "pmdco:PMD_0020133",
    "id": "mass-specimen-316l-tensile-bar-1",
    "quality_of": "specimen-316l-tensile-bar-1",
    "specified_by_value": {
      "type": "obi:0001931",
      "id": "mass-svs-specimen-316l-tensile-bar-1",
      "value": 50.3,
      "unit": "uo:0000021"
    }
  },
  "has_composition": {
    "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/chemical-composition/PMDCo/#v1.0.0",
    "type": "pmdco:PMD_0000551",
    "id": "chem-comp-316l-tensile-bar-1",
    "quality_of": "specimen-316l-tensile-bar-1",
    "is_subject_of": {
      "type": "pmdco:PMD_0025002",
      "id": "chem-comp-316l-tensile-bar-1-spec",
      "has_member": [
        {
          "type": "pmdco:PMD_0025997",
          "id": "frac

---
## Step 3: Convert to RDF

The OO-LD document is parsed as JSON-LD using the ontology context from
`specs/schema.oold.yaml`, which maps every field name to its ontology IRI.
rdflib ≥ 7 handles JSON-LD 1.1 natively.

In [7]:
flat = specimen_schema.parse(oold_doc, base=BASE)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 126 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/chem-comp-316l-tensile-bar-1> a pmdco:PMD_0000551 ;
    obo:RO_0000080 <https://example.org/specimen-316l-tensile-bar-1> ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/chemical-composition/PMDCo/#v1.0.0> ;
    pmdco:PMD_0000004 <https://example.org/chem-comp-316l-tensile-bar-1-spec> .

<https://example.org/chem-comp-316l-tensile-bar-1-spec> a pmdco:PMD_0025002 ;
    obo:RO_0002351 <https://example.org/frac-C>,
        <https://example.org/frac-Cr>,
        <https://example.org/frac-Fe>,
        <https://example.org/frac-Mn>,
        <https://example.org/frac-Mo>,
        <https://example.org/frac-N>,
        <https://example.org/frac-Ni>,
   

In [8]:
# Optional: save to file
out_ttl = HERE / "output_specimen.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_specimen.ttl


---
## Step 4: Validate against the SHACL shapes

The graph is checked against two SHACL shape files: one for the specimen
(mass, label) and one for the chemical composition (element fractions,
proportions):

| Shape file | Validates |
|---|---|
| `specimen/PMDCo/specs/shape.ttl` | Specimen label, Mass value and unit |
| `chemical-composition/PMDCo/specs/shape.ttl` | ChemicalComposition, element fractions |

> `inference="rdfs"` is required because some shapes target superclasses
> (e.g. `Proportion`) and rely on RDFS reasoning to match their subtypes.

In [9]:
conforms, violations = specimen_schema.validate(flat, also=[comp_schema])

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


---
## Step 5: Inspect the graph

Quick checks to confirm the key values were captured correctly.

In [10]:
BFO   = rdflib.Namespace("http://purl.obolibrary.org/obo/BFO_")
RO    = rdflib.Namespace("http://purl.obolibrary.org/obo/RO_")
PMDCO = rdflib.Namespace("https://w3id.org/pmd/co/")
OBI   = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")
IAO   = rdflib.Namespace("http://purl.obolibrary.org/obo/IAO_")

# The specimen has two has_quality links (mass + composition).
# Identify the Mass node by its RDF type.
specimen_iri = next(flat.subjects(rdflib.RDF.type, BFO["0000030"]))
label        = flat.value(specimen_iri, rdflib.RDFS.label)
mass_quality = next(
    q for q in flat.objects(specimen_iri, RO["0000086"])
    if (q, rdflib.RDF.type, PMDCO["PMD_0020133"]) in flat
)
mass_svs  = flat.value(mass_quality, PMDCO["PMD_0000077"])
mass_val  = flat.value(mass_svs, OBI["0001937"])
mass_unit = flat.value(mass_svs, IAO["0000039"])

print(f"Specimen : {specimen_iri}")
print(f"Label    : {label}")
print(f"Mass     : {mass_val} {str(mass_unit).rsplit('/', 1)[-1]}")

Specimen : https://example.org/specimen-316l-tensile-bar-1
Label    : 316L Tensile Bar #1
Mass     : 50.3 UO_0000021


In [11]:
SPARQL_ELEMENTS = """
PREFIX ro:    <http://purl.obolibrary.org/obo/RO_>
PREFIX pmdco: <https://w3id.org/pmd/co/>
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>

SELECT ?elemType ?fracValue ?unit
WHERE {
  ?spec a pmdco:PMD_0025002 ;
        ro:0002351 ?frac .
  ?frac a pmdco:PMD_0025997 ;
        obi:0001937 ?fracValue ;
        iao:0000039 ?unit ;
        pmdco:PMD_hasFractionElement ?elemNode .
  ?elemNode a ?elemType .
  FILTER(?elemType != pmdco:PMD_0025997)
}
ORDER BY DESC(?fracValue)
"""

rows = list(flat.query(SPARQL_ELEMENTS))
print(f"{'Element (PMDCo IRI fragment)':<22}  {'Value':>8}  Unit")
print("-" * 50)
for r in rows:
    frag = str(r.elemType).rsplit("/", 1)[-1]
    unit = str(r.unit).rsplit("/", 1)[-1]
    print(f"{frag:<22}  {float(r.fracValue):>8.3f}  {unit}")

Element (PMDCo IRI fragment)     Value  Unit
--------------------------------------------------
PMD_0020026               65.345  UO_0000163
PMD_0020029               17.000  UO_0000163
PMD_0020051               12.000  UO_0000163
PMD_0020034                2.500  UO_0000163
PMD_0020078                1.800  UO_0000163
PMD_0020050                0.500  UO_0000163
PMD_0020038                0.100  UO_0000163
PMD_0020047                0.045  UO_0000163
PMD_0020059                0.030  UO_0000163
PMD_0020030                0.030  UO_0000163


---
## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with specimen name, mass, and element fractions |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against SHACL shapes |
| 5 | Key values are extracted from the graph to confirm correctness |

To use your own specimen, edit `docs/example.input.json` and re-run all cells.
`mass_value` / `mass_unit` are optional; omit them if the mass is not known.

---

## Further reading

- [Chemical Composition — PMDCo](../../../chemical-composition/PMDCo/README.md): the composition sub-schema used here
- [OO-LD primer](../../../../docs/2_oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../../docs/3_schema-format.md): how to write your own schema